<span></span>
<center>
<h1><b>Fall 2026<br/>
CSC-372 Project Part 8<br/></b></h1>

# Attention

</center>


<a href="https://colab.research.google.com/github/fahadsultan/csc372/blob/main/Project7.ipynb" target="_blank" align="right">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<!-- <img src="Project7_2.png" width="100%"> -->


In [ ]:
!wget https://github.com/karpathy/build-nanogpt/blob/master/input.txt

--2026-10-28 08:08:55--  https://github.com/karpathy/build-nanogpt/blob/master/input.txt
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: 'input.txt'

input.txt               [ <=>                ] 206.44K  1.03MB/s    in 0.2s    

2026-10-28 08:08:57 (1.03 MB/s) - 'input.txt' saved [211397]



In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F


# Configuration
torch.manual_seed(42)

BATCH_SIZE      = 32
CONTEXT_LENGTH  = 64
EMBEDDING_DIM   = 64
LEARNING_RATE   = 1e-3
TRAINING_STEPS  = 2000
EVAL_INTERVAL   = 100
EVAL_BATCHES    = 20

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"


# Data preparation
text = Path("input.txt").read_text(encoding="utf-8")
characters = sorted(set(text))
vocabulary_size = len(characters)

character_to_id = {character: i for i, character in enumerate(characters)}
id_to_character = {i: character for character, i in character_to_id.items()}


def encode(text):
    return [character_to_id[character] for character in text]

def decode(ids):
    return "".join(id_to_character[i] for i in ids)


data = torch.tensor(encode(text), dtype=torch.long)
split_index = int(0.9 * len(data))

datasets = {
    "train": data[:split_index],
    "validation": data[split_index:],
}

if vocabulary_size < 2:
    raise ValueError("The corpus must contain at least two characters.")

for name, dataset in datasets.items():
    if len(dataset) <= CONTEXT_LENGTH:
        raise ValueError(
            f"The {name} split needs more than "
            f"{CONTEXT_LENGTH} characters."
        )



In [ ]:

def get_batch(split):
    """Return inputs and next-character targets, both shaped (B, T)."""
    dataset = datasets[split]

    starts = torch.randint(
        len(dataset) - CONTEXT_LENGTH,
        (BATCH_SIZE,),
    )

    inputs = torch.stack([
        dataset[start:start + CONTEXT_LENGTH]
        for start in starts
    ])

    targets = torch.stack([
        dataset[start + 1:start + CONTEXT_LENGTH + 1]
        for start in starts
    ])

    return inputs.to(DEVICE), targets.to(DEVICE)


In [ ]:
class CausalSelfAttention(nn.Module):
    """One attention head, implemented explicitly."""

    def __init__(self, embedding_dim, context_length):
        super().__init__()

        self.query              = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.key                = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.value              = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.output_projection  = nn.Linear(embedding_dim, embedding_dim)

        # True marks a permitted connection, including the diagonal.
        allowed = torch.ones(context_length, context_length, dtype=torch.bool).tril()

        self.register_buffer("allowed", allowed)

    def forward(self, x):
        # x: (B, T, C)
        _, length, channels = x.shape

        queries = self.query(x)  # (B, T, C)
        keys    = self.key(x)       # (B, T, C)
        values  = self.value(x)   # (B, T, C)

        # Each query is compared with every key in its sequence.
        scores = queries @ keys.transpose(-2, -1)  # (B, T, T)
        scores = scores / math.sqrt(channels)

        # Mask future positions BEFORE softmax.
        scores = scores.masked_fill(~self.allowed[:length, :length], float("-inf"),)

        weights = F.softmax(scores, dim=-1)  # (B, T, T)
        attended = weights @ values         # (B, T, C)

        return self.output_projection(attended), weights


In [ ]:
class AttentionLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocabulary_size, EMBEDDING_DIM)
        self.position_embedding = nn.Embedding(CONTEXT_LENGTH, EMBEDDING_DIM)
        self.attention_norm     = nn.LayerNorm(EMBEDDING_DIM)
        self.attention          = CausalSelfAttention(EMBEDDING_DIM, CONTEXT_LENGTH)
        self.final_norm         = nn.LayerNorm(EMBEDDING_DIM)
        self.language_head      = nn.Linear(EMBEDDING_DIM, vocabulary_size)

    def forward(self, token_ids, targets=None):
        # token_ids: (B, T)
        _, length = token_ids.shape

        if length > CONTEXT_LENGTH:
            raise ValueError("Input exceeds the context length.")

        positions = torch.arange(length, device=token_ids.device)

        x = (self.token_embedding(token_ids) + self.position_embedding(positions))  # (B, T, C)

        update, weights = self.attention(self.attention_norm(x))
        x = x + update  # Residual connection

        logits = self.language_head(self.final_norm(x))  # (B, T, vocabulary_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, vocabulary_size),targets.reshape(-1),)

        return logits, loss, weights

    @torch.no_grad()
    def generate(self, token_ids, new_characters=300):
        was_training = self.training
        self.eval()

        for _ in range(new_characters):
            # Retain only the supported context window.
            context = token_ids[:, -CONTEXT_LENGTH:]
            logits, _, _ = self(context)

            # The final position predicts the next character.
            probabilities = F.softmax(
                logits[:, -1, :], dim=-1
            )
            next_id = torch.multinomial(
                probabilities, num_samples=1
            )

            token_ids = torch.cat(
                [token_ids, next_id], dim=1
            )

        self.train(was_training)
        return token_ids

In [ ]:
@torch.no_grad()
def estimate_losses(model):
    was_training = model.training
    model.eval()
    results = {}

    for split in datasets:
        losses = []

        for _ in range(EVAL_BATCHES):
            inputs, targets = get_batch(split)
            _, loss, _ = model(inputs, targets)
            losses.append(loss.item())

        results[split] = sum(losses) / len(losses)

    model.train(was_training)
    return results


@torch.no_grad()
def check_causality(model):
    """Changing future inputs must not change earlier logits."""
    was_training = model.training
    model.eval()

    inputs, _ = get_batch("validation")
    changed = inputs.clone()
    cutoff = CONTEXT_LENGTH // 2

    changed[:, cutoff:] = (
        changed[:, cutoff:] + 1
    ) % vocabulary_size

    original_logits, _, weights = model(inputs)
    changed_logits, _, _ = model(changed)

    torch.testing.assert_close(
        original_logits[:, :cutoff],
        changed_logits[:, :cutoff],
        atol=1e-5,
        rtol=1e-5,
    )

    # Every attention weight above the diagonal must be zero.
    assert torch.count_nonzero(
        torch.triu(weights, diagonal=1)
    ).item() == 0

    model.train(was_training)
    print("Causality check passed.")





# New

In [2]:
# download input text file

!wget https://raw.githubusercontent.com/karpathy/build-nanogpt/refs/heads/master/input.txt

--2026-10-28 13:44:06--  https://raw.githubusercontent.com/karpathy/build-nanogpt/refs/heads/master/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: 'input.txt'

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.07s   

2026-10-28 13:44:06 (14.4 MB/s) - 'input.txt' saved [1115394/1115394]



In [3]:
# read input
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

#convert to utf-8
encoded = text.encode("utf-8")

In [4]:
len(encoded), len(set(encoded))

(1115394, 65)

In [ ]:
from collections import Counter


def merge_pair(tokens, pair, new_id):
    """Replace non-overlapping occurrences of pair with new_id."""
    result = []
    i = 0
    while i < len(tokens):
        if (i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == pair):
            result.append(new_id)
            i += 2
        else:
            result.append(tokens[i])
            i += 1

    return result


def byte_pair_encoding(encoded, vocab_size):
    encoded = list(encoded)

    if not encoded:
        return encoded

    # Track the vocabulary separately from tokens still in the sequence.
    current_vocab_size = len(set(encoded))
    new_id = max(encoded) + 1

    while current_vocab_size < vocab_size:
        # print("Current vocabulary size:", current_vocab_size,"Target vocabulary size:", vocab_size,)

        pairs = Counter(zip(encoded, encoded[1:]))

        if not pairs:
            break  # No adjacent tokens remain to merge.

        pair, count = pairs.most_common(1)[0]
        # print("Merging:", pair, "Count:", count, "New ID:", new_id)

        encoded = merge_pair(encoded, pair, new_id)

        new_id += 1
        current_vocab_size += 1

    return encoded


vocab_size = 80
compressed = byte_pair_encoding(encoded, vocab_size)

Current vocabulary size: 65 Target vocabulary size: 80
Merging: (101, 32) Count: 27643 New ID: 123
Current vocabulary size: 66 Target vocabulary size: 80
Merging: (116, 104) Count: 22739 New ID: 124
Current vocabulary size: 67 Target vocabulary size: 80
Merging: (116, 32) Count: 16508 New ID: 125
Current vocabulary size: 68 Target vocabulary size: 80
Merging: (115, 32) Count: 15364 New ID: 126
Current vocabulary size: 69 Target vocabulary size: 80
Merging: (100, 32) Count: 14165 New ID: 127
Current vocabulary size: 70 Target vocabulary size: 80
Merging: (44, 32) Count: 14098 New ID: 128
Current vocabulary size: 71 Target vocabulary size: 80
Merging: (111, 117) Count: 12730 New ID: 129
Current vocabulary size: 72 Target vocabulary size: 80
Merging: (101, 114) Count: 11771 New ID: 130
Current vocabulary size: 73 Target vocabulary size: 80
Merging: (105, 110) Count: 10606 New ID: 131
Current vocabulary size: 74 Target vocabulary size: 80
Merging: (121, 32) Count: 10283 New ID: 132
Current